In [ ]:
using ChaosTools
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
λ = 0.6

ξ_d = 3.3
ν_d = 1

Q = Inf
t_max=1000.0

β = 0.5

environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
drive_prepared = prepare(drive)

x_0, p_0 = 0.0, 0.0
system = SystemMichiel(β, λ, x_0, p_0, 1)
environment_prepared = prepare(system,environment)

dt = 1 / (34*sqrt(2))

In [ ]:
β_list = range(start = 0.342, stop = 2.941, step = 0.0223)
starting_points_list = [(.5, .5), (1.0,1.0), (5.0,1.0), (1.0,5.0), (6.0,0.0), (8.0,0.0), (9.0,0.0), (10.0,0.0)]

t_max=100
Lyap_exp_dict = Dict{Tuple{Float64,Float64}, Vector{Float64}}()
t_w = 0
for starting_point in starting_points_list
    x_0, p_0 = starting_point
    Lyap_exp = Vector{Float64}()
    for β in β_list
        system = SystemMichiel(β, λ, x_0, p_0, 1)
        system_prepared = prepare(system)
                
        params = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, [p_0, x_0], params)

        push!(Lyap_exp, lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w))  
    end
    Lyap_exp_dict[starting_point] = Lyap_exp
end

In [ ]:
p0 = plot()
for starting_point in starting_points_list
    Lyap_exp = Lyap_exp_dict[starting_point]
    plot!(β_list, Lyap_exp, label="")
end
plot!(xlabel=L"\beta", ylabel="Lyapunov exponent")
display(p0)